# 🧠 คำอธิบายและตัวอย่างการปฏิบัติการการถดถอยริดจ์ (Ridge Regression - L2 Regularization)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **การถดถอยริดจ์ (Ridge Regression)**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างชุดข้อมูลจำลองขนาดเล็กแบบมีสัญญาณรบกวน (Noise) และฟิตด้วยพหุนามดีกรีสูง
2. สังเกตว่าการค้นหาด้วยเส้นถดถอยแบบธรรมดา (Ordinary Least Squares: OLS) เผชิญกับปัญหาโอเวอร์ฟิตอย่างรุนแรง (Overfitting - ความแปรปรวนสูง) และค่าสัมประสิทธิ์มีขนาดใหญ่เกินจริงอย่างไร
3. ประยุกต์ใช้ **Ridge Regression (L2 Regularization)** เพื่อปรับลดค่าน้ำหนักที่ใหญ่เกินไป
4. จำลองภาพผลกระทบของค่าความเข้มข้นของการจัดระเบียบ (Regularization Strength: $\lambda$) ว่าช่วยปรับความโค้งของฟังก์ชันให้ราบเรียบและป้องกันการโอเวอร์ฟิตได้อย่างไร
5. ลงมือพัฒนาตัวทำนาย Ridge Regression จากศูนย์โดยใช้ **สมการปกติที่ถูกดัดแปลง (Modified Normal Equation)**:
   $$\mathbf{w} = (\mathbf{X}^T \mathbf{X} + \lambda \mathbf{I}')^{-1} \mathbf{X}^T \mathbf{y}$$
   (โดยกำหนดให้ $\mathbf{I}'$ คือเมทริกซ์เอกลักษณ์ที่ตั้งค่า $I'_{0,0} = 0$ เพื่อไม่ให้ลงโทษเทอมจุดตัดแกน Y/อคติ)

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(10)

## 1. การสร้างข้อมูลจำลอง (Data Generation)

เราจะสุ่มสร้างจุดข้อมูลสำหรับการฝึกสอนเพียง **15 จุด** เท่านั้น โดยใช้ฟังก์ชันคลื่นคลื่นโคไซน์:
$$y = \cos(1.5 \pi x) + \epsilon$$

การฟิตโมเดลพหุนามดีกรีสูง (เช่น ดีกรี 10) บนจุดข้อมูลเพียง 15 จุด จะเป็นต้นเหตุที่ทำให้วิธีการค้นหาแบบธรรมดา (OLS) เกิดปัญหาโอเวอร์ฟิตอย่างเห็นได้ชัด

In [ ]:
# สร้างข้อมูลสำหรับการฝึกสอน (15 จุด)
X_train = np.sort(np.random.rand(15, 1) * 2 - 1, axis=0)
y_train = np.cos(1.5 * np.pi * X_train) + np.random.randn(15, 1) * 0.15

# สร้างข้อมูลสำหรับการทดสอบ (30 จุด) สำหรับใช้ประเมินผล
X_test = np.sort(np.random.rand(30, 1) * 2 - 1, axis=0)
y_test = np.cos(1.5 * np.pi * X_test) + np.random.randn(30, 1) * 0.15

# สร้างตาราง Grid ความถี่สูงเพื่อใช้ลากเส้นกราฟให้มีความโค้งราบเรียบสวยงาม
X_grid = np.linspace(-1.1, 1.1, 200).reshape(-1, 1)

# พล็อตกราฟแสดงข้อมูลสำหรับฝึกสอนและทดสอบ
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='blue', label='Train Data (15 points)')
plt.scatter(X_test, y_test, color='red', alpha=0.5, label='Test Data')
plt.plot(X_grid, np.cos(1.5 * np.pi * X_grid), color='gray', linestyle='--', label='True function')
plt.title('Small Noisy Dataset for Overfitting Demonstration')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. การเกิดปัญหาโอเวอร์ฟิตจากการถดถอยพหุนามแบบ OLS

เรามาลองดูกันว่าจะเกิดอะไรขึ้นเมื่อนำพหุนามดีกรี 10 มาฟิตกับจุดข้อมูลเทรน 15 จุด ด้วยวิธีการถดถอยเชิงเส้นธรรมดา (OLS)

In [ ]:
# แปลงคุณลักษณะให้เป็นดีกรี 10
degree = 10
poly = PolynomialFeatures(degree=degree, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_grid_poly = poly.transform(X_grid)

# เทรนแบบจำลอง OLS
ols_model = LinearRegression()
ols_model.fit(X_train_poly, y_train)

# ทำนายผล
y_grid_pred_ols = ols_model.predict(X_grid_poly)

# พล็อตกราฟ
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='blue', label='Train Data')
plt.plot(X_grid, y_grid_pred_ols, color='red', linewidth=2.5, label='OLS Fit (Degree 10)')
plt.plot(X_grid, np.cos(1.5 * np.pi * X_grid), color='gray', linestyle='--', label='True Function')
plt.ylim(-2.5, 2.5)
plt.title('OLS Overfitting: Massive Oscillations')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# แสดงค่าสัมประสิทธิ์ที่โมเดลเรียนรู้ได้
print("Learned coefficients for OLS:")
print("Intercept:", ols_model.intercept_[0])
for idx, w in enumerate(ols_model.coef_[0]):
    print(f"w_{idx+1} (X^{idx+1}): {w:.4f}")

## 3. การทำ Regularization ด้วย Ridge Regression (L2 Penalty)

สังเกตว่าสัมประสิทธิ์บางตัวของ OLS มีค่าสูงมากอย่างรุนแรง (บางตัวขึ้นไปถึงระดับหลักพันหรือหลักล้าน) ซึ่งเป็นอาการเด่นของปัญหาโอเวอร์ฟิต (Overfitting)
ต่อจากนี้ เราจะทำการฟิตข้อมูลด้วยพหุนามดีกรี 10 ตัวเดิม แต่เปลี่ยนมาใช้แบบจำลอง **Ridge Regression** ที่มีการปรับเปลี่ยนระดับความแรงของพารามิเตอร์การจัดระเบียบ $\alpha$ (ซึ่งเป็นสัญลักษณ์แทน $\lambda$)

In [ ]:
# แบบจำลอง Ridge ที่มีค่าความแรงอัลฟ่า (alpha) ต่างๆ กัน
alphas = [1e-5, 0.01, 1.0, 100.0]

plt.figure(figsize=(12, 6))
plt.scatter(X_train, y_train, color='blue', label='Train Data')

for alpha in alphas:
    ridge_model = Ridge(alpha=alpha)
    ridge_model.fit(X_train_poly, y_train)
    
    y_grid_pred = ridge_model.predict(X_grid_poly)
    
    # คำนวณหาผลรวมกำลังสองของน้ำหนัก (Sum of squared weights)
    w_squared_sum = np.sum(ridge_model.coef_ ** 2)
    
    plt.plot(X_grid, y_grid_pred, label=f'Ridge (α={alpha}, ∑w²={w_squared_sum:.2f})', linewidth=2)

plt.plot(X_grid, np.cos(1.5 * np.pi * X_grid), color='gray', linestyle='--', label='True Function')
plt.ylim(-1.5, 1.5)
plt.title('Ridge Regression: Shrunk Coefficients Prevent Overfitting')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. การสร้าง Ridge Regression จากศูนย์ด้วยสมการปกติ (Normal Equation with L2 Penalty from Scratch)

สูตรสมการปกติสำหรับการถดถอยแบบริดจ์คือ:
$$\mathbf{w} = (\mathbf{X}^T \mathbf{X} + \lambda \mathbf{I}')^{-1} \mathbf{X}^T \mathbf{y}$$

โดยที่:
*   $\lambda$ คือพารามิเตอร์โทษปรับ (Penalty Parameter)
*   $\mathbf{I}'$ คือเมทริกซ์เอกลักษณ์ที่ดัดแปลง เนื่องจากเราจะไม่ปรับโทษค่าน้ำหนักอคติ (bias intercept) ค่าในคอลัมน์แรกแถวแรกสุดจึงถูกกำหนดให้เป็น 0:
    $$\mathbf{I}' = \begin{bmatrix} 0 & 0 & 0 & \dots \\ 0 & 1 & 0 & \dots \\ 0 & 0 & 1 & \dots \\ \vdots & \vdots & \vdots & \ddots \end{bmatrix}$$

เรามาลองเขียนฟังก์ชันคุณลักษณะพหุนามและแก้สมการนี้ด้วย NumPy กันครับ

In [ ]:
def get_polynomial_features(X, degree):
    m = len(X)
    X_poly = np.ones((m, 1))
    for power in range(1, degree + 1):
        X_poly = np.hstack((X_poly, X ** power))
    return X_poly

def solve_ridge_normal_equation(X, y, lmbda):
    """
    หาค่า w = (X.T @ X + lmbda * I')^-1 @ X.T @ y
    """
    n_features = X.shape[1]
    
    # สร้างเมทริกซ์เอกลักษณ์ดัดแปลง (กำหนดให้คอลัมน์แรกแถวแรกมีค่าเป็น 0 เพื่อยกเว้นอคติ)
    I_prime = np.identity(n_features)
    I_prime[0, 0] = 0.0
    
    # คำนวณองค์ประกอบต่างๆ ตามสูตรสมการปกติ
    left_side = X.T @ X + lmbda * I_prime
    left_side_inv = np.linalg.inv(left_side)
    w = left_side_inv @ X.T @ y
    return w

# แปลง X_train ให้เป็นดีกรี 10 โดยรวมเทอมอคติ (bias) เข้าไปในคอลัมน์แรกเรียบร้อยแล้ว
X_train_scratch = get_polynomial_features(X_train, degree=10)
X_grid_scratch = get_polynomial_features(X_grid, degree=10)

# กำหนดค่าแลมบ์ดา (lambda)
lmbda = 0.01

# แก้สมการเพื่อหาน้ำหนัก
weights_scratch = solve_ridge_normal_equation(X_train_scratch, y_train, lmbda)

# ทำนายผลลัพธ์
y_grid_pred_scratch = X_grid_scratch @ weights_scratch

# ประเมินผลความคลาดเคลื่อนบนชุดข้อมูลฝึกสอนเทียบกับข้อมูลทดสอบ
X_test_scratch = get_polynomial_features(X_test, degree=10)
y_train_pred = X_train_scratch @ weights_scratch
y_test_pred = X_test_scratch @ weights_scratch

print(f"Scratch Ridge Model (λ={lmbda}) Performance:")
print(f"Train MSE: {mean_squared_error(y_train, y_train_pred):.4f}")
print(f"Test MSE:  {mean_squared_error(y_test, y_test_pred):.4f}")

# พล็อตกราฟแสดงเส้นพหุนามริดจ์ที่สร้างขึ้นเองจากศูนย์
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='blue', label='Train Data')
plt.plot(X_grid, y_grid_pred_scratch, color='green', linewidth=2.5, label=f'Scratch Ridge (λ={lmbda})')
plt.plot(X_grid, np.cos(1.5 * np.pi * X_grid), color='gray', linestyle='--', label='True Function')
plt.ylim(-1.5, 1.5)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 💡 ความเชื่อมโยงสู่ Deep Learning และ YOLO
*   **การลดลงของน้ำหนัก (Weight Decay):** ในการปรับปรุงค่าเครือข่ายประสาทเชิงลึก (เช่น การฝึกสอนแบบจำลอง YOLO) การจัดระเบียบแบบ L2 จะถูกขนานนามว่า **Weight Decay** โดยกระบวนการนี้จะทำการบวกสัดส่วนเล็กน้อยของน้ำหนักเข้าไปในค่าเกรเดียนต์ในแต่ละขั้นตอนการอัปเดต เพื่อบีบบังคับตัวหาค่าเหมาะสมที่สุด (Optimizer) ไม่ให้เรียนรู้น้ำหนักที่มีขนาดใหญ่เกินจริง ส่งผลให้โครงข่ายประสาทเรียนรู้ฟีเจอร์พารามิเตอร์ที่ราบเรียบไม่ซับซ้อน แทนการเพ่งเล็งไปยังพิกเซลหรือลวดลายเฉพาะจุด ซึ่งช่วยลดปัญหา Overfitting อย่างได้ผล และช่วยยกระดับความแม่นยำในการระบุประเภทวัตถุในสภาพแวดล้อมจริงที่มีสิ่งรบกวน